## Parte 1 — Softmax con temperatura

La softmax con temperatura se implementa de forma numéricamente estable restando el máximo de los logits antes de exponenciar: p_i = exp((z_i − m)/T) / Σ_j exp((z_j − m)/T), con m = max_j z_j. Esto evita desbordamientos cuando T es pequeño, pues los exponentes quedan siempre ≤ 0. Sobre los logits [2.0, 1.0, 0.5, 0.2, −1.0, −3.0] se obtienen las siguientes distribuciones (cada una suma 1.0) y entropías en bits:

| Token | T = 0.5 | T = 1.0 | T = 2.0 |
|-------|---------|---------|---------|
| t0 | 0.8231 | 0.5516 | 0.3583 |
| t1 | 0.1114 | 0.2029 | 0.2173 |
| t2 | 0.0410 | 0.1231 | 0.1693 |
| t3 | 0.0225 | 0.0912 | 0.1457 |
| t4 | 0.0020 | 0.0275 | 0.0800 |
| t5 | 0.0000 | 0.0037 | 0.0294 |
| **Entropía (bits)** | **0.9147** | **1.7998** | **2.2888** |

Se verifica que las tres distribuciones suman 1.0 y que la entropía es monótonamente creciente en T: 0.9147 bits (T = 0.5) < 1.7998 bits (T = 1) < 2.2888 bits (T = 2). Al bajar la temperatura la masa se concentra en t0 (0.8231 en T = 0.5 frente a 0.3583 en T = 2) y t5 se vuelve despreciable (0.0000 en T = 0.5); al subirla, la probabilidad se reparte hacia los tokens de baja probabilidad.

In [1]:
# T1: código aprobado por el crítico
# T1: Softmax con temperatura (NumPy, estable), distribuciones para T=0.5, 1, 2,
# entropía en bits y tabla con cuatro decimales.

import json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt


# ---------- Datos del enunciado ----------
tokens = ["t0", "t1", "t2", "t3", "t4", "t5"]
logits = np.array([2.0, 1.0, 0.5, 0.2, -1.0, -3.0], dtype=float)
temperaturas = [0.5, 1.0, 2.0]


# ---------- Implementación ----------
def softmax_con_temperatura(z, T):
    """p_i = exp(z_i/T) / sum_j exp(z_j/T), numéricamente estable.

    Se resta el máximo de los logits escalados antes de exponenciar para
    evitar desbordamiento (overflow) en exp.
    """
    z = np.asarray(z, dtype=float)
    T = float(T)
    if T <= 0:
        raise ValueError("La temperatura debe ser positiva.")
    z_esc = z / T
    z_esc = z_esc - np.max(z_esc)  # estabilidad numérica
    e = np.exp(z_esc)
    return e / np.sum(e)


def entropia_bits(p):
    """H = -sum p_i * log2(p_i), en bits (se ignoran p_i = 0)."""
    p = np.asarray(p, dtype=float)
    p = p[p > 0]
    return float(-np.sum(p * np.log2(p)))


# ---------- Cálculos ----------
distribuciones = {T: softmax_con_temperatura(logits, T) for T in temperaturas}
entropias = {T: entropia_bits(distribuciones[T]) for T in temperaturas}
sumas = {T: float(np.sum(distribuciones[T])) for T in temperaturas}
monotona = bool(entropias[0.5] < entropias[1.0] < entropias[2.0])


# ---------- Tabla con cuatro decimales ----------
tabla = pd.DataFrame(
    {f"T={T}": [f"{p:.4f}" for p in distribuciones[T]] + [f"{entropias[T]:.4f}"]
     for T in temperaturas},
    index=tokens + ["Entropía (bits)"],
)


# ---------- Figura ----------
fig, axes = plt.subplots(1, 3, figsize=(12, 4), sharey=True)
for ax, T in zip(axes, temperaturas):
    ax.bar(tokens, distribuciones[T], color="steelblue", edgecolor="black")
    ax.set_title(f"T = {T}   (H = {entropias[T]:.4f} bits)")
    ax.set_ylim(0, 1)
    ax.set_xlabel("Token")
    ax.grid(axis="y", alpha=0.3)
axes[0].set_ylabel("Probabilidad")
fig.suptitle("Softmax con temperatura sobre logits [2.0, 1.0, 0.5, 0.2, -1.0, -3.0]")
fig.tight_layout()
plt.savefig("t1_softmax_temperatura.png", dpi=120)
plt.close(fig)


# ---------- resultados.json (contrato) ----------
resultados = {
    "logits": {t: float(z) for t, z in zip(tokens, logits)},
    "temperaturas": temperaturas,
    "distribuciones": {
        f"T={T}": {t: float(p) for t, p in zip(tokens, distribuciones[T])}
        for T in temperaturas
    },
    "entropias_bits": {f"T={T}": entropias[T] for T in temperaturas},
    "suma_distribuciones": {f"T={T}": sumas[T] for T in temperaturas},
    "entropia_monotona_creciente": monotona,
    "tabla_4_decimales": {
        "filas": tokens + ["entropia_bits"],
        "columnas": [f"T={T}" for T in temperaturas],
        "valores": (
            [[f"{distribuciones[T][i]:.4f}" for T in temperaturas] for i in range(len(tokens))]
            + [[f"{entropias[T]:.4f}" for T in temperaturas]]
        ),
    },
}
with open("resultados.json", "w", encoding="utf-8") as f:
    json.dump(resultados, f, ensure_ascii=False, indent=2)


# ---------- Resumen ----------
print("Tabla (4 decimales):")
print(tabla.to_string())
print("\nSuma de cada distribución:", {f"T={T}": round(s, 12) for T, s in sumas.items()})
print("Entropías (bits):", {f"T={T}": round(h, 6) for T, h in entropias.items()})
print("Entropía monótona creciente con T:", monotona)
print("Archivos generados: resultados.json, t1_softmax_temperatura.png")


Tabla (4 decimales):
                  T=0.5   T=1.0   T=2.0
t0               0.8231  0.5516  0.3583
t1               0.1114  0.2029  0.2173
t2               0.0410  0.1231  0.1693
t3               0.0225  0.0912  0.1457
t4               0.0020  0.0275  0.0800
t5               0.0000  0.0037  0.0294
Entropía (bits)  0.9147  1.7998  2.2888

Suma de cada distribución: {'T=0.5': 1.0, 'T=1.0': 1.0, 'T=2.0': 1.0}
Entropías (bits): {'T=0.5': 0.914682, 'T=1.0': 1.799836, 'T=2.0': 2.288826}
Entropía monótona creciente con T: True
Archivos generados: resultados.json, t1_softmax_temperatura.png


## Parte 2 — Muestreo de núcleo (top-p)

Sobre la distribución de T = 1 de la Parte 1 se ordenan los tokens por probabilidad descendente y se acumula hasta alcanzar p = 0.9. Tras t2 la acumulada es 0.8776 (< 0.9), y tras t3 alcanza 0.9688 (≥ 0.9), por lo que el conjunto mínimo que supera el umbral tiene k = 4 tokens: el núcleo es {t0, t1, t2, t3}, con masa total 0.9688. Los tokens t4 y t5 quedan fuera del núcleo y reciben probabilidad 0; al renormalizar (dividiendo entre 0.9688), la distribución resultante suma 1.0:

| Token | Prob. original (T = 1) | Acumulada ordenada | En núcleo | Prob. renormalizada |
|-------|------------------------|--------------------|-----------|---------------------|
| t0 | 0.5516 | 0.5516 | sí | 0.5694 |
| t1 | 0.2029 | 0.7546 | sí | 0.2095 |
| t2 | 0.1231 | 0.8776 | sí | 0.1270 |
| t3 | 0.0912 | 0.9688 | sí | 0.0941 |
| t4 | 0.0275 | 0.9963 | no | 0.0000 |
| t5 | 0.0037 | 1.0000 | no | 0.0000 |

El efecto de top-p es recortar la cola de baja probabilidad (t4 y t5, la masa restante hasta 1.0000 fuera del núcleo) y redistribuir su peso proporcionalmente entre los tokens supervivientes: t0 pasa de 0.5516 a 0.5694, t1 de 0.2029 a 0.2095, t2 de 0.1231 a 0.1270 y t3 de 0.0912 a 0.0941.

In [2]:
# T2: código aprobado por el crítico
# -*- coding: utf-8 -*-
"""
T2 — Muestreo de núcleo (top-p) con p = 0.9 sobre la distribución de T = 1.

- Carga la distribución de T = 1 obtenida en la Parte 1 (entrada/T1/resultados.json);
  si no estuviera disponible, la recalcula con la softmax numéricamente estable.
- Aplica top-p con p = 0.9: ordena los tokens por probabilidad descendente,
  conserva el conjunto MÍNIMO cuya probabilidad acumulada alcanza 0.9 y renormaliza.
- Escribe resultados.json (contrato), guarda fig_T2_top_p.png e imprime un resumen.
"""

import json
from pathlib import Path

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# ----------------------------------------------------------------------
# Parámetros fijos del enunciado
# ----------------------------------------------------------------------
LOGITS = np.array([2.0, 1.0, 0.5, 0.2, -1.0, -3.0])
TOKENS = [f"t{i}" for i in range(LOGITS.size)]
TEMPERATURA = 1.0
P = 0.9

# ----------------------------------------------------------------------
# 1) Distribución de T = 1 de la Parte 1 (resultado previo T1)
# ----------------------------------------------------------------------
ruta_t1 = Path("entrada") / "T1" / "resultados.json"
origen = None
probs = None
if ruta_t1.exists():
    with ruta_t1.open("r", encoding="utf-8") as f:
        res_t1 = json.load(f)
    dists = res_t1.get("distribuciones", {})
    for clave in ("T=1.0", "T=1"):
        if clave in dists:
            probs = np.array([float(dists[clave][t]) for t in TOKENS])
            origen = f"{ruta_t1} (clave '{clave}')"
            break
if probs is None:
    # Respaldo: softmax con temperatura, numéricamente estable (resta del máximo)
    z = LOGITS / TEMPERATURA
    z = z - z.max()
    ez = np.exp(z)
    probs = ez / ez.sum()
    origen = "recalculada con softmax estable (T = 1)"

# ----------------------------------------------------------------------
# 2) Top-p con p = 0.9: conjunto mínimo con acumulada >= p
# ----------------------------------------------------------------------
orden = np.argsort(-probs, kind="stable")      # índices de mayor a menor probabilidad
probs_ord = probs[orden]
acum_ord = np.cumsum(probs_ord)

# Número mínimo de tokens para alcanzar p: el token que cruza el umbral SE incluye
k = int(np.searchsorted(acum_ord, P, side="left")) + 1
k = min(k, probs.size)

nucleo_idx = orden[:k]
fuera_idx = orden[k:]

en_nucleo = np.zeros(probs.size, dtype=bool)
en_nucleo[nucleo_idx] = True

probs_nucleo = np.where(en_nucleo, probs, 0.0)   # los tokens fuera del núcleo quedan en 0
suma_nucleo = float(probs_nucleo.sum())
probs_renorm = probs_nucleo / suma_nucleo        # renormalización

tokens_nucleo = [TOKENS[i] for i in nucleo_idx]
tokens_fuera = [TOKENS[i] for i in fuera_idx]
acum_nucleo = float(acum_ord[k - 1])

# ----------------------------------------------------------------------
# 3) Verificaciones del criterio de éxito
# ----------------------------------------------------------------------
suma_renorm = float(probs_renorm.sum())
alcanza_p = bool(acum_nucleo >= P)
es_minimo = bool(k == 1 or acum_ord[k - 2] < P)          # sin el último token no se alcanza p
fuera_en_cero = bool(np.all(probs_renorm[~en_nucleo] == 0.0))
suma_exacta = bool(abs(suma_renorm - 1.0) < 1e-12)

# ----------------------------------------------------------------------
# 4) Figura
# ----------------------------------------------------------------------
x = np.arange(probs.size)
ancho = 0.38
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11.5, 4.6))

ax1.bar(x - ancho / 2, probs, width=ancho, color="#8d99ae", label="Original (T = 1)")
ax1.bar(x + ancho / 2, probs_renorm, width=ancho,
        color=["#2a9d8f" if m else "#e76f51" for m in en_nucleo],
        label="Renormalizada (top-p, p = 0.9)")
ax1.set_xticks(x)
ax1.set_xticklabels(TOKENS)
ax1.set_ylabel("Probabilidad")
ax1.set_ylim(0, 0.65)
ax1.set_title("Distribución original vs. renormalizada\n(verde = núcleo, rojo = fuera del núcleo)")
ax1.legend(fontsize=8)

ax2.plot(range(1, probs.size + 1), acum_ord, marker="o", color="#264653",
         label="Acumulada (orden descendente)")
ax2.axhline(P, color="#e63946", linestyle="--", label=f"p = {P}")
ax2.axvline(k, color="#2a9d8f", linestyle=":", label=f"k mínimo = {k}")
ax2.set_xticks(range(1, probs.size + 1))
ax2.set_xticklabels([TOKENS[i] for i in orden])
ax2.set_xlabel("Tokens ordenados por probabilidad descendente")
ax2.set_ylabel("Probabilidad acumulada")
ax2.set_title(f"Acumulada del núcleo: {acum_nucleo:.4f} ≥ {P}")
ax2.legend(fontsize=8)

fig.suptitle("T2 — Muestreo de núcleo (top-p) con p = 0.9 sobre la distribución de T = 1",
             fontsize=11)
fig.tight_layout(rect=[0, 0, 1, 0.93])
fig.savefig("fig_T2_top_p.png", dpi=120)
plt.close(fig)

# ----------------------------------------------------------------------
# 5) resultados.json (contrato de la subtarea)
# ----------------------------------------------------------------------
tabla_filas = []
for r, idx in enumerate(orden, start=1):
    tabla_filas.append([
        TOKENS[idx],
        f"{probs[idx]:.4f}",
        f"{acum_ord[r - 1]:.4f}",
        "sí" if en_nucleo[idx] else "no",
        f"{probs_renorm[idx]:.4f}",
    ])

resultados = {
    "subtarea": "T2 — Muestreo de núcleo (top-p) con p = 0.9 sobre la distribución de T = 1",
    "p": P,
    "temperatura": TEMPERATURA,
    "fuente_distribucion_T1": origen,
    "distribucion_original_T1": {t: float(v) for t, v in zip(TOKENS, probs)},
    "orden_descendente": [TOKENS[i] for i in orden],
    "probs_ordenadas": [float(v) for v in probs_ord],
    "acumuladas_ordenadas": [float(v) for v in acum_ord],
    "k_tokens_minimo": k,
    "tokens_nucleo": tokens_nucleo,
    "prob_acumulada_nucleo": acum_nucleo,
    "tokens_fuera_nucleo": tokens_fuera,
    "distribucion_renormalizada": {t: float(v) for t, v in zip(TOKENS, probs_renorm)},
    "suma_distribucion_renormalizada": suma_renorm,
    "verificaciones": {
        "acumulada_nucleo_alcanza_p": alcanza_p,
        "conjunto_es_minimo": es_minimo,
        "tokens_fuera_con_probabilidad_cero": fuera_en_cero,
        "suma_renormalizada_es_1": suma_exacta,
    },
    "tabla_4_decimales": {
        "columnas": ["token", "prob_original_T1", "acumulada_ordenada",
                     "en_nucleo", "prob_renormalizada"],
        "filas": tabla_filas,
    },
    "figura": "fig_T2_top_p.png",
}

with open("resultados.json", "w", encoding="utf-8") as f:
    json.dump(resultados, f, ensure_ascii=False, indent=2)

# ----------------------------------------------------------------------
# 6) Resumen por consola
# ----------------------------------------------------------------------
print("=" * 66)
print("T2 — Top-p (nucleus sampling) con p = 0.9 sobre la distribución T = 1")
print("=" * 66)
print(f"Fuente de la distribución T = 1 : {origen}")
print("\nTokens ordenados por probabilidad descendente:")
for r, idx in enumerate(orden, start=1):
    marca = "NÚCLEO" if en_nucleo[idx] else "fuera "
    print(f"  {r}. {TOKENS[idx]}  p = {probs[idx]:.6f}  "
          f"acum = {acum_ord[r - 1]:.6f}  [{marca}]")
print(f"\nNúcleo mínimo (k = {k}): {tokens_nucleo}")
print(f"Probabilidad acumulada del núcleo: {acum_nucleo:.6f}  (>= {P}: {alcanza_p})")
print(f"Tokens fuera del núcleo: {tokens_fuera}  (quedan con probabilidad 0)")
print("\nDistribución renormalizada:")
for t, v in zip(TOKENS, probs_renorm):
    print(f"  {t}: {v:.6f}")
print(f"Suma de la distribución renormalizada: {suma_renorm:.12f}")
print("\nVerificaciones:", resultados["verificaciones"])
print("Figura guardada: fig_T2_top_p.png | Contrato escrito: resultados.json")


T2 — Top-p (nucleus sampling) con p = 0.9 sobre la distribución T = 1
Fuente de la distribución T = 1 : entrada/T1/resultados.json (clave 'T=1.0')

Tokens ordenados por probabilidad descendente:
  1. t0  p = 0.551623  acum = 0.551623  [NÚCLEO]
  2. t1  p = 0.202931  acum = 0.754553  [NÚCLEO]
  3. t2  p = 0.123084  acum = 0.877637  [NÚCLEO]
  4. t3  p = 0.091183  acum = 0.968820  [NÚCLEO]
  5. t4  p = 0.027464  acum = 0.996283  [fuera ]
  6. t5  p = 0.003717  acum = 1.000000  [fuera ]

Núcleo mínimo (k = 4): ['t0', 't1', 't2', 't3']
Probabilidad acumulada del núcleo: 0.968820  (>= 0.9: True)
Tokens fuera del núcleo: ['t4', 't5']  (quedan con probabilidad 0)

Distribución renormalizada:
  t0: 0.569376
  t1: 0.209462
  t2: 0.127045
  t3: 0.094117
  t4: 0.000000
  t5: 0.000000
Suma de la distribución renormalizada: 1.000000000000

Verificaciones: {'acumulada_nucleo_alcanza_p': True, 'conjunto_es_minimo': True, 'tokens_fuera_con_probabilidad_cero': True, 'suma_renormalizada_es_1': True}
Fig

## Parte 3 — Comprobación empírica

Se tomaron 10 000 muestras de la distribución renormalizada de la Parte 2 con numpy.random.default_rng(0). Las frecuencias observadas reproducen de cerca las probabilidades teóricas; t4 y t5, con probabilidad teórica 0.0000, no reciben ninguna muestra:

| Token | Prob. teórica | Frecuencia observada | Conteo |
|-------|---------------|----------------------|--------|
| t0 | 0.5694 | 0.5682 | 5682 |
| t1 | 0.2095 | 0.2093 | 2093 |
| t2 | 0.1270 | 0.1296 | 1296 |
| t3 | 0.0941 | 0.0929 | 929 |
| t4 | 0.0000 | 0.0000 | 0 |
| t5 | 0.0000 | 0.0000 | 0 |

La divergencia KL de la empírica respecto de la teórica, KL(q‖p) = Σ_i q_i · log2(q_i / p_i) (los términos con q_i = 0 aportan 0), es de 5.0069e-05 bits. Este valor, prácticamente cero, confirma que el muestreador con la semilla fijada genera muestras cuya distribución empírica es estadísticamente indistinguible de la teórica: la mayor discrepancia absoluta se da en t2 (0.1296 observado frente a 0.1270 teórico), del orden esperable por fluctuación de muestreo con n = 10 000.

In [3]:
# T3: código aprobado por el crítico
import json
from pathlib import Path

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# ---------------------------------------------------------------
# T3 — Comprobación empírica
# 1) Cargar la distribución renormalizada (top-p p=0.9, T=1) de T2
# 2) Tomar 10,000 muestras con numpy.random.default_rng(0)
# 3) Gráfico de barras: frecuencias observadas vs probabilidades teóricas
# 4) Divergencia KL en bits de la empírica respecto de la teórica
# ---------------------------------------------------------------

N = 10000
SEED = 0
FIGURA = "fig_T3_frecuencias_vs_teoricas.png"

# --- 1) Distribución teórica: preferir el resultado de T2; si no existe, recalcular ---
ruta_t2 = Path("entrada/T2/resultados.json")
fuente = ""
if ruta_t2.exists():
    with open(ruta_t2, "r", encoding="utf-8") as f:
        res_t2 = json.load(f)
    dist_renorm = res_t2["distribucion_renormalizada"]
    fuente = "entrada/T2/resultados.json (clave 'distribucion_renormalizada')"
else:
    # Respaldo determinista: softmax(T=1) + top-p p=0.9 sobre los logits del enunciado
    tokens_fb = ["t0", "t1", "t2", "t3", "t4", "t5"]
    logits = np.array([2.0, 1.0, 0.5, 0.2, -1.0, -3.0], dtype=float)
    z = logits - logits.max()
    p_t1 = np.exp(z) / np.exp(z).sum()
    orden = np.argsort(-p_t1, kind="stable")
    acum = np.cumsum(p_t1[orden])
    k = int(np.searchsorted(acum, 0.9) + 1)  # conjunto mínimo que alcanza 0.9
    nucleo = orden[:k]
    p_ren = np.zeros_like(p_t1)
    p_ren[nucleo] = p_t1[nucleo] / p_t1[nucleo].sum()
    dist_renorm = {t: float(p) for t, p in zip(tokens_fb, p_ren)}
    fuente = "recalculada desde logits (softmax T=1, top-p p=0.9)"

tokens = sorted(dist_renorm.keys())  # t0..t5
p_teorica = np.array([dist_renorm[t] for t in tokens], dtype=float)

# --- 2) Muestreo: 10,000 muestras con default_rng(0) ---
rng = np.random.default_rng(SEED)
indices = rng.choice(len(tokens), size=N, p=p_teorica)
conteos = np.bincount(indices, minlength=len(tokens))
frecuencias = conteos / N

# --- 3) Divergencia KL en bits: KL(empírica || teórica) = Σ q_i log2(q_i / p_i) ---
# Los tokens con p_teorica = 0 (t4, t5) no reciben muestras (q_i = 0 => término 0).
mask = frecuencias > 0
kl_bits = float(np.sum(frecuencias[mask] * np.log2(frecuencias[mask] / p_teorica[mask])))

# --- 4) Gráfico de barras: observado vs teórico por token ---
x = np.arange(len(tokens))
ancho = 0.38
fig, ax = plt.subplots(figsize=(8.5, 5.2))
b1 = ax.bar(x - ancho / 2, p_teorica, width=ancho, color="#4C72B0",
            edgecolor="black", label="Probabilidad teórica (top-p renormalizada)")
b2 = ax.bar(x + ancho / 2, frecuencias, width=ancho, color="#DD8452",
            edgecolor="black", label=f"Frecuencia observada (n={N})")
for barras in (b1, b2):
    for rect in barras:
        h = rect.get_height()
        ax.annotate(f"{h:.4f}",
                    xy=(rect.get_x() + rect.get_width() / 2, h),
                    xytext=(0, 3), textcoords="offset points",
                    ha="center", va="bottom", fontsize=8)
ax.set_xticks(x)
ax.set_xticklabels(tokens)
ax.set_xlabel("Token")
ax.set_ylabel("Probabilidad / Frecuencia")
ax.set_title("Parte 3 — Frecuencias observadas vs probabilidades teóricas\n"
             f"top-p p=0.9 (T=1), {N} muestras, default_rng({SEED}), KL = {kl_bits:.6f} bits")
ax.legend()
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(FIGURA, dpi=120)
plt.close(fig)

# --- 5) Resultados (contrato) ---
tabla_4dec = [[t, f"{p:.4f}", f"{q:.4f}", str(int(c))]
              for t, p, q, c in zip(tokens, p_teorica, frecuencias, conteos)]

resultados = {
    "subtarea": "T3 — Comprobación empírica: muestreo de la distribución renormalizada (top-p p=0.9, T=1) y divergencia KL",
    "n_muestras": N,
    "semilla": SEED,
    "generador": "numpy.random.default_rng(0)",
    "fuente_distribucion_teorica": fuente,
    "tokens": tokens,
    "probabilidades_teoricas": {t: float(p) for t, p in zip(tokens, p_teorica)},
    "conteos_observados": {t: int(c) for t, c in zip(tokens, conteos)},
    "frecuencias_observadas": {t: float(q) for t, q in zip(tokens, frecuencias)},
    "divergencia_KL_bits": kl_bits,
    "KL_bits": kl_bits,
    "definicion_KL": "KL(empírica || teórica) = Σ_i q_i * log2(q_i / p_i); términos con q_i = 0 aportan 0",
    "tabla_4_decimales": {
        "columnas": ["token", "prob_teorica", "frecuencia_observada", "conteo"],
        "filas": tabla_4dec,
    },
    "figura": FIGURA,
    "verificaciones": {
        "suma_probabilidades_teoricas": float(p_teorica.sum()),
        "suma_frecuencias_observadas": float(frecuencias.sum()),
        "total_muestras": int(conteos.sum()),
        "tokens_sin_muestras": [t for t, c in zip(tokens, conteos) if c == 0],
    },
}

with open("resultados.json", "w", encoding="utf-8") as f:
    json.dump(resultados, f, ensure_ascii=False, indent=2)

# --- Resumen breve ---
print("=== T3 — Comprobación empírica ===")
print(f"Fuente de la distribución teórica: {fuente}")
print(f"Muestras: {N} | Generador: numpy.random.default_rng({SEED})")
print(f"{'Token':<6}{'Teórica':>12}{'Observada':>12}{'Conteo':>10}")
for t, p, q, c in zip(tokens, p_teorica, frecuencias, conteos):
    print(f"{t:<6}{p:>12.6f}{q:>12.6f}{c:>10d}")
print(f"Divergencia KL (bits), KL(empírica || teórica): {kl_bits:.8f}")
print(f"Figura guardada: {FIGURA}")
print("Resultados escritos en resultados.json")


=== T3 — Comprobación empírica ===
Fuente de la distribución teórica: entrada/T2/resultados.json (clave 'distribucion_renormalizada')
Muestras: 10000 | Generador: numpy.random.default_rng(0)
Token      Teórica   Observada    Conteo
t0        0.569376    0.568200      5682
t1        0.209462    0.209300      2093
t2        0.127045    0.129600      1296
t3        0.094117    0.092900       929
t4        0.000000    0.000000         0
t5        0.000000    0.000000         0
Divergencia KL (bits), KL(empírica || teórica): 0.00005007
Figura guardada: fig_T3_frecuencias_vs_teoricas.png
Resultados escritos en resultados.json


## Parte 4 — Pregunta conceptual

**T → 0:** al dividir los logits por una temperatura cada vez más pequeña, las diferencias (z_i − z_max)/T se hacen infinitamente negativas para todo token distinto del de mayor logit, de modo que exp de esos términos tiende a 0 y toda la masa se concentra en el argmax (t0, con logit 2.0). La distribución degenera en una masa de punto y su entropía tiende a 0 bits. Las entropías medidas en la Parte 1 muestran exactamente esa tendencia: al reducir T de 2 a 1 y de 1 a 0.5, la entropía cae de 2.2888 a 1.7998 y luego a 0.9147 bits, mientras la probabilidad de t0 sube de 0.3583 a 0.5516 y a 0.8231 y la de t5 cae de 0.0294 a 0.0037 y a 0.0000. **T → ∞:** los logits divididos por T tienden todos a 0, los exponenciales se igualan y la distribución tiende a la uniforme sobre los seis tokens, que es la distribución de entropía máxima posible para ese vocabulario; la entropía medida crece hacia ese límite (0.9147 → 1.7998 → 2.2888 bits) y las probabilidades se aplanan, con t0 bajando y t5 subiendo a medida que T aumenta.

**Por qué T → 0 equivale a la decodificación voraz:** la decodificación greedy selecciona siempre el token de mayor logit, es decir, el argmax de la distribución. Cuando T → 0, la softmax asigna probabilidad 1 a ese token y 0 a todos los demás, de modo que muestrear de esa distribución produce el argmax con probabilidad 1: ambos procedimientos generan exactamente la misma secuencia. La evidencia de la Parte 1 es la caída monótona de la entropía al disminuir T (2.2888 → 1.7998 → 0.9147 bits) junto con la concentración de la masa en t0 (0.3583 → 0.5516 → 0.8231): en el límite T → 0 la entropía alcanza su mínimo (0 bits) y no queda incertidumbre alguna, que es precisamente la situación del muestreo greedy, donde la "distribución" efectiva es un determinismo sobre el token más probable.